In [9]:
import pandas as pd
from pathlib import Path
from ifc.config import load_data_config
from ifc.config import resolve_project_root

5) Eda Outlier mechanism diagnosis (ratio explosion check) 
The target variable `revenue_change` is defined as a year-over-year percentage change:

revenue_change = ((production_value_t − production_value_{t−1}) / production_value_{t−1}) × 100

As a consequence, extreme target values may arise not only from large absolute changes
in revenue, but also from very small values of the denominator (prior-year revenue).



In [14]:
data_cfg = load_data_config()

train_path, test_path = data_cfg.resolve_paths()

print(train_path)

C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\train_data.csv


In [15]:
df = pd.read_csv(train_path)

In [16]:
df = df.sort_values(["company_id", "fiscal_year"]).copy()


We sort observations by company and fiscal year to ensure that lagged values
correctly correspond to the previous year within each firm.


In [17]:
df["production_value_lag1"] = (
    df.groupby("company_id")["production_value"]
      .shift(1)
)


This variable reconstructs the implicit denominator of the target formula,
namely the production value observed in the previous fiscal year (t−1).


In [18]:
df["abs_revenue_change"] = df["revenue_change"].abs()


In [19]:
top20 = (
    df.sort_values("abs_revenue_change", ascending=False)
      .loc[:, [
          "company_id",
          "fiscal_year",
          "production_value_lag1",
          "production_value",
          "revenue_change"
      ]]
      .head(20)
)

top20


,company_id,fiscal_year,production_value_lag1,production_value,revenue_change
9793,COMP_02485,2021,3.351168e+07,1.012812e+11,302126.48
380,COMP_00096,2020,4.856124e+06,1.034987e+10,213030.29
5876,COMP_01490,2021,9.048074e+06,1.232272e+10,136091.65
8571,COMP_02174,2019,2.363747e+07,1.752024e+10,74020.61
11557,COMP_02932,2019,3.080814e+07,1.893038e+10,61346.04
3543,COMP_00896,2021,1.363047e+08,7.208317e+10,52783.86
1765,COMP_00450,2020,7.906439e+07,2.926382e+10,36912.64
11471,COMP_02910,2020,7.870927e+08,2.762358e+11,34995.72
1840,COMP_00469,2019,4.681431e+08,1.570978e+11,33457.64
2959,COMP_00749,2020,3.897862e+07,1.275458e+10,32622.00


The table reports the 20 observations with the largest absolute values of
`revenue_change`, together with current and lagged production values.



In [20]:
# Define near-zero denominator as bottom 5%
p5 = df["production_value_lag1"].dropna().quantile(0.05)

# Define extreme targets as top 1% of |revenue_change|
thr = df["abs_revenue_change"].quantile(0.99)

extreme = df["abs_revenue_change"] >= thr
near_zero = df["production_value_lag1"] < p5

share_near_zero = (extreme & near_zero).sum() / extreme.sum()

p5, thr, share_near_zero


(np.float64(78369336.73200001),
 np.float64(6313.31359999999),
 np.float64(0.33707865168539325))

We quantify the mechanism by measuring the share of extreme target observations
(top 1% in absolute value) whose prior-year production value lies in the bottom 5%
of the distribution.


Takeaway (EDA 5)
Extreme values of `revenue_change` are largely consistent with denominator effects:
when prior-year production value is very small, the ratio defining the target
can explode even for moderate absolute changes. This explains the presence of
heavy-tailed outliers and motivates the use of robust modeling strategies
(e.g., winsorization or target transformations).


 6) ATECO segmentation (The "K-Shape" Check)
After identifying strong temporal shocks in the target distribution,
we investigate whether these shocks affected economic sectors asymmetrically.


In [21]:
sector_summary = (
    df.groupby("ateco_sector")
      .agg(
          count=("revenue_change", "size"),
          median_revenue_change=("revenue_change", "median"),
          p95_revenue_change=("revenue_change", lambda s: s.quantile(0.95)),
      )
      .sort_values("count", ascending=False)
      .head(10)
)

sector_summary


,count,median_revenue_change,p95_revenue_change
ateco_sector,,,
46,1688,-6.87,1860.3130
47,1491,6.64,1575.0750
41,1465,8.41,2041.5165
43,1189,10.17,1467.3020
25,1095,4.93,1775.8380
10,958,-3.04,1635.1770
62,950,14.68,1870.4100
56,754,-2.72,1686.3490
45,662,1.32,1745.5250


This table summarizes target behavior across the most represented ATECO sectors.


In [22]:
sector_year_pivot = df.pivot_table(
    index="ateco_sector",
    columns="fiscal_year",
    values="revenue_change",
    aggfunc="median"
)

sector_year_pivot


fiscal_year,2019,2020,2021
ateco_sector,,,
10,2.785,-22.820,21.710
25,19.040,1.125,-8.700
41,6.580,-8.890,26.290
43,6.370,-8.570,34.025
45,-33.130,18.850,-2.130
46,-15.795,9.980,-8.580
47,2.340,17.900,3.230
56,-8.385,-32.530,34.890
62,33.200,7.335,2.705


Median revenue changes are reported by sector and fiscal year to highlight
heterogeneous shock and recovery patterns.


Takeaway (EDA 6) 
Target volatility differs substantially across ATECO sectors, and the impact
of the 2020 shock is clearly asymmetric. Some sectors experience sharp contractions
followed by recovery, while others remain relatively stable, confirming the
presence of a K-shaped dynamic. This supports sector identity as a key predictor
for modeling revenue changes during the COVID period.
